In [1]:
%pip install azureml-widgets -q
%pip install azureml-train-automl-runtime==1.57.0 -q
%pip install --upgrade azureml-sdk[notebooks,automl] -q

StatementMeta(02dc0817-3613-4306-b25f-f5a8bf6c7202, 0, 12, Finished, Available, Finished)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
azureml-opendatasets 1.47.0 requires azureml-core~=1.47.0, but you have azureml-core 1.57.0.post1 which is incompatible.
azureml-opendatasets 1.47.0 requires azureml-telemetry~=1.47.0, but you have azureml-telemetry 1.57.0 which is incompatible.
azureml-mlflow 1.47.0 requires azure-storage-blob<=12.13.0,>=12.5.0, but you have azure-storage-blob 12.14.1 which is incompatible.
azure-identity 1.7.0 requires msal-extensions~=0.3.0, but you have msal-extensions 1.0.0 which is incompatible.
You should consider upgrading via the '/nfs4/pyenv-7ed94a6e-1e8d-4fbd-9bd4-37aa23a3775a/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [2]:
from azureml.core import Workspace, Experiment
session_id = "267722"
subscription_id= "cdbe0b43-92a0-4715-838a-f2648cc7ad21"
resource_group= f"aml-quickstarts-{session_id}"
workspace_name= f"quick-starts-ws-{session_id}"

#ws = Workspace.from_config()
ws = Workspace.get(name=workspace_name, subscription_id=subscription_id, resource_group=resource_group)
exp = Experiment(workspace=ws, name="udacity-project")

print('Workspace name: ' + ws.name, 
      'Azure region: ' + ws.location, 
      'Subscription id: ' + ws.subscription_id, 
      'Resource group: ' + ws.resource_group, sep = '\n')

run = exp.start_logging()

StatementMeta(02dc0817-3613-4306-b25f-f5a8bf6c7202, 0, 18, Finished, Available, Finished)

Performing interactive authentication. Please follow the instructions on the terminal.
To sign in, use a web browser to open the page https://microsoft.com/devicelogin and enter the code DMH4JRBPD to authenticate.
Interactive authentication successfully completed.
Workspace name: quick-starts-ws-267722
Azure region: westeurope
Subscription id: cdbe0b43-92a0-4715-838a-f2648cc7ad21
Resource group: aml-quickstarts-267722


In [3]:
from azureml.core.compute import ComputeTarget, AmlCompute
cluster_name = "project-1"
vm_size="Standard_D2_V2"
compute_config=AmlCompute.provisioning_configuration(vm_size=vm_size,max_nodes=4)
# TODO: Create compute cluster
# Use vm_size = "Standard_D2_V2" in your provisioning configuration.
# max_nodes should be no greater than 4.
### YOUR CODE HERE ###
from azureml.exceptions import ComputeTargetException
try:
    compute_target=ComputeTarget(workspace=ws,name=cluster_name)
    print('Found existing compute cluster:',cluster_name)
except ComputeTargetException:
    compute_target=ComputeTarget.create(workspace=ws,name=cluster_name,provisioning_configuration=compute_config)
    compute_target.wait_for_completion(show_output=True)


StatementMeta(02dc0817-3613-4306-b25f-f5a8bf6c7202, 0, 19, Finished, Available, Finished)

InProgress..
SucceededProvisioning operation finished, operation "Succeeded"
Succeeded
AmlCompute wait for completion finished

Minimum number of nodes requested have been provisioned


In [4]:
from azureml.widgets import RunDetails
from azureml.train.sklearn import SKLearn
from azureml.train.hyperdrive.run import PrimaryMetricGoal
from azureml.train.hyperdrive.policy import BanditPolicy
from azureml.train.hyperdrive.sampling import RandomParameterSampling
from azureml.train.hyperdrive.runconfig import HyperDriveConfig
from azureml.train.hyperdrive.parameter_expressions import choice, uniform
from azureml.core import Environment, ScriptRunConfig
import os

# Specify parameter sampler
ps = RandomParameterSampling(parameter_space={'--C':uniform(0.5,1.5),'--max_iter':choice(16,32,64,128)})

# Specify a Policy
policy = BanditPolicy(slack_factor=0.2)

if "training" not in os.listdir():
    os.mkdir("./training")

# Setup environment for your training run
sklearn_env = Environment.from_conda_specification(
    name='sklearn-env', file_path=f'Users/odl_user_{session_id}/conda_dependencies.yml')

# Create a ScriptRunConfig Object to specify the configuration details of your training job
src = ScriptRunConfig(
    source_directory='.',
    script=f'Users/odl_user_{session_id}/train.py',
    environment=sklearn_env,compute_target=compute_target)

# Create a HyperDriveConfig using the src object, hyperparameter sampler, and policy.
hyperdrive_config = HyperDriveConfig(
    hyperparameter_sampling=ps,
    primary_metric_goal=PrimaryMetricGoal.MAXIMIZE,
    primary_metric_name='Accuracy',
    policy=policy,
    run_config=src,
    max_total_runs=15,
    max_concurrent_runs=4)

StatementMeta(02dc0817-3613-4306-b25f-f5a8bf6c7202, 0, 20, Finished, Available, Finished)

In [5]:
# Submit your hyperdrive run to the experiment and show run details with the widget.

### YOUR CODE HERE ###
exp_run=exp.submit(hyperdrive_config)
RunDetails(exp_run).show()
exp_run.wait_for_completion(show_output=True)

StatementMeta(02dc0817-3613-4306-b25f-f5a8bf6c7202, 0, 21, Submitted, Running, Running)

Failed to load image Python extension: libc10_cuda.so: cannot open shared object file: No such file or directory


Initializing logging file for interpret-community


_HyperDriveWidget(widget_settings={'childWidgetDisplay': 'popup', 'send_telemetry': False, 'log_level': 'INFO'…

RunId: HD_8bd44cbb-12ae-44d1-94a2-27fee9e35cfe
Web View: https://ml.azure.com/runs/HD_8bd44cbb-12ae-44d1-94a2-27fee9e35cfe?wsid=/subscriptions/cdbe0b43-92a0-4715-838a-f2648cc7ad21/resourcegroups/aml-quickstarts-267722/workspaces/quick-starts-ws-267722&tid=660b3398-b80e-49d2-bc5b-ac1dc93b5254

Streaming azureml-logs/hyperdrive.txt

[2024-09-17T15:57:50.820635][GENERATOR][INFO]Trying to sample '4' jobs from the hyperparameter space
[2024-09-17T15:57:51.3655178Z][SCHEDULER][INFO]Scheduling job, id='HD_8bd44cbb-12ae-44d1-94a2-27fee9e35cfe_0' 
[2024-09-17T15:57:51.5344558Z][SCHEDULER][INFO]Scheduling job, id='HD_8bd44cbb-12ae-44d1-94a2-27fee9e35cfe_1' 
[2024-09-17T15:57:51.7092490Z][SCHEDULER][INFO]Scheduling job, id='HD_8bd44cbb-12ae-44d1-94a2-27fee9e35cfe_2' 
[2024-09-17T15:57:51.670670][GENERATOR][INFO]Successfully sampled '4' jobs, they will soon be submitted to the execution target.
[2024-09-17T15:57:51.7340201Z][SCHEDULER][INFO]Scheduling job, id='HD_8bd44cbb-12ae-44d1-94a2-27fee9e35c

{'runId': 'HD_8bd44cbb-12ae-44d1-94a2-27fee9e35cfe',
 'target': 'project-1',
 'status': 'Completed',
 'startTimeUtc': '2024-09-17T15:57:49.182074Z',
 'endTimeUtc': '2024-09-17T16:18:00.191747Z',
 'services': {},
 'properties': {'primary_metric_config': '{"name":"Accuracy","goal":"maximize"}',
  'resume_from': 'null',
  'runTemplate': 'HyperDrive',
  'azureml.runsource': 'hyperdrive',
  'platform': 'AML',
  'ContentSnapshotId': '6f03029f-f4a2-49fe-bf15-15f484c885ab',
  'user_agent': 'python/3.10.6 (Linux-4.15.0-1178-azure-x86_64-with-glibc2.27) msrest/0.6.21 Hyperdrive.Service/1.0.0 Hyperdrive.SDK/core.1.57.0',
  'space_size': 'infinite_space_size',
  'best_child_run_id': 'HD_8bd44cbb-12ae-44d1-94a2-27fee9e35cfe_9',
  'score': '0.9083459787556905',
  'best_metric_status': 'Succeeded',
  'best_data_container_id': 'dcid.HD_8bd44cbb-12ae-44d1-94a2-27fee9e35cfe_9'},
 'inputDatasets': [],
 'outputDatasets': [],
 'runDefinition': {'configuration': None,
  'attribution': None,
  'telemetryValu

In [6]:
import joblib
# Get your best run and save the model from that run.

### YOUR CODE HERE ###
best_run=exp_run.get_best_run_by_primary_metric()
print(best_run)

StatementMeta(, , , Waiting, , Waiting)

Run(Experiment: udacity-project,
Id: HD_8bd44cbb-12ae-44d1-94a2-27fee9e35cfe_9,
Type: azureml.scriptrun,
Status: Completed)


In [7]:
#print(best_run.get_details())
print(best_run.get_metrics())
print(best_run.get_file_names())

StatementMeta(, , , Waiting, , Waiting)

{'Regularization Strength:': 1.1042391231776982, 'Max iterations:': 64, 'Accuracy': 0.9083459787556905}
['logs/azureml/dataprep/0/backgroundProcess.log', 'logs/azureml/dataprep/0/backgroundProcess_Telemetry.log', 'logs/azureml/dataprep/0/rslex.log.2024-09-17-16', 'system_logs/cs_capability/cs-capability.log', 'system_logs/hosttools_capability/hosttools-capability.log', 'system_logs/lifecycler/execution-wrapper.log', 'system_logs/lifecycler/lifecycler.log', 'system_logs/metrics_capability/metrics-capability.log', 'system_logs/snapshot_capability/snapshot-capability.log', 'user_logs/std_log.txt']


In [8]:
joblib.dump(best_run.get_metrics(),'best_run.json')

StatementMeta(, , , Waiting, , Waiting)

['best_run.json']

In [9]:
from azureml.data.dataset_factory import TabularDatasetFactory

# Create TabularDataset using TabularDatasetFactory
# Data is available at: 
# "https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv"

### YOUR CODE HERE ###
url = "https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv"
ds = TabularDatasetFactory.from_delimited_files(path=url)

StatementMeta(, , , Waiting, , Waiting)

In [10]:
#from Users.odl_user_267614.train import clean_data

from sklearn.linear_model import LogisticRegression
import argparse
import os
import numpy as np
from sklearn.metrics import mean_squared_error
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import pandas as pd
from azureml.core.run import Run
from azureml.data.dataset_factory import TabularDatasetFactory

def clean_data(data):
    # Dict for cleaning data
    months = {"jan":1, "feb":2, "mar":3, "apr":4, "may":5, "jun":6, "jul":7, "aug":8, "sep":9, "oct":10, "nov":11, "dec":12}
    weekdays = {"mon":1, "tue":2, "wed":3, "thu":4, "fri":5, "sat":6, "sun":7}

    # Clean and one hot encode data
    #x_df = pd.DataFrame(data)
    #x_df = x_df.dropna()
    print(data)
    x_df = data.to_pandas_dataframe().dropna()
    jobs = pd.get_dummies(x_df.job, prefix="job")
    x_df.drop("job", inplace=True, axis=1)
    x_df = x_df.join(jobs)
    x_df["marital"] = x_df.marital.apply(lambda s: 1 if s == "married" else 0)
    x_df["default"] = x_df.default.apply(lambda s: 1 if s == "yes" else 0)
    x_df["housing"] = x_df.housing.apply(lambda s: 1 if s == "yes" else 0)
    x_df["loan"] = x_df.loan.apply(lambda s: 1 if s == "yes" else 0)
    contact = pd.get_dummies(x_df.contact, prefix="contact")
    x_df.drop("contact", inplace=True, axis=1)
    x_df = x_df.join(contact)
    education = pd.get_dummies(x_df.education, prefix="education")
    x_df.drop("education", inplace=True, axis=1)
    x_df = x_df.join(education)
    x_df["month"] = x_df.month.map(months)
    x_df["day_of_week"] = x_df.day_of_week.map(weekdays)
    x_df["poutcome"] = x_df.poutcome.apply(lambda s: 1 if s == "success" else 0)

    y_df = x_df.pop("y").apply(lambda s: 1 if s == "yes" else 0)
    return x_df, y_df
# Use the clean_data function to clean your data.
x, y = clean_data(ds)

StatementMeta(, , , Waiting, , Waiting)

TabularDataset
{
  "definition": "EnginelessDataflow:\n---\ntype: mltable\npaths:\n  - pattern: \"wasbs://automl-sample-notebook-data@automlsamplenotebookdata.blob.core.windows.net/bankmarketing_train.csv\"\ntransformations:\n  - read_delimited:\n      path_column: Path\n      include_path_column: false\n      encoding: utf8\n      support_multi_line: false\n      delimiter: \",\"\n      empty_as_string: false\n      partition_size: 20971520\n      header: all_files_same_headers\n      infer_column_types: false\n  - convert_column_types:\n      - columns: duration\n        column_type: int\n      - columns: contact\n        column_type: string\n      - columns: education\n        column_type: string\n      - columns: poutcome\n        column_type: string\n      - columns: day_of_week\n        column_type: string\n      - columns: pdays\n        column_type: int\n      - columns: month\n        column_type: string\n      - columns: loan\n        column_type: string\n      - columns: eur

In [43]:
from sklearn.model_selection import train_test_split
import pandas as pd
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
training_data = x_train
training_data['y']=y_train

StatementMeta(02dc0817-3613-4306-b25f-f5a8bf6c7202, 0, 59, Finished, Available, Finished)

In [51]:
datastore = ws.get_default_datastore()
import os
os.makedirs('local')
pd.DataFrame(training_data).to_csv('local/training_data.csv')

StatementMeta(02dc0817-3613-4306-b25f-f5a8bf6c7202, 0, 67, Finished, Available, Finished)

In [52]:
from azureml.core import Dataset
Dataset.File.upload_directory(src_dir='local/',target=datastore)
my_data = Dataset.Tabular.from_delimited_files(path=[(datastore, ('training_data.csv'))])

StatementMeta(02dc0817-3613-4306-b25f-f5a8bf6c7202, 0, 68, Finished, Available, Finished)

Validating arguments.
Arguments validated.
'overwrite' is set to False. Any file already present in the target will be skipped.'
Uploading files from '/synfs/notebook/0/aml_notebook_mount/local' to '/'
Copying 1 files with concurrency set to 1
Copied /synfs/notebook/0/aml_notebook_mount/local/training_data.csv, file 1 out of 1. Destination path: https://mlstrg267722.blob.core.windows.net/azureml-blobstore-918e7a4c-cbc3-412a-9c31-c3a195f5cb1c/training_data.csv
Files copied=1, skipped=0, failed=0
Creating new dataset


In [54]:
from azureml.train.automl import AutoMLConfig

# Set parameters for AutoMLConfig
# NOTE: DO NOT CHANGE THE experiment_timeout_minutes PARAMETER OR YOUR INSTANCE WILL TIME OUT.
# If you wish to run the experiment longer, you will need to run this notebook in your own
# Azure tenant, which will incur personal costs.
automl_config = AutoMLConfig(
    experiment_timeout_minutes=30,
    compute_target=compute_target,
    task="classification",
    primary_metric="accuracy",
    training_data=my_data, max_concurrent_iterations=4,
    label_column_name='y',
    n_cross_validations=4)

StatementMeta(02dc0817-3613-4306-b25f-f5a8bf6c7202, 0, 70, Finished, Available, Finished)

In [55]:
# Submit your automl run

### YOUR CODE HERE ###
from azureml.core.experiment import Experiment
experiment = Experiment(ws, "project-1-automl")
autml_run = experiment.submit(config=automl_config, show_output=True)
RunDetails(autml_run).show()

StatementMeta(02dc0817-3613-4306-b25f-f5a8bf6c7202, 0, 71, Submitted, Running, Running)

Submitting remote run.
No run_configuration provided, running on project-1 with default configuration
Running on remote compute: project-1


Experiment,Id,Type,Status,Details Page,Docs Page
project-1-automl,AutoML_ddf0b2e0-1279-416c-befa-1d0c05b36ec6,automl,NotStarted,Link to Azure Machine Learning studio,Link to Documentation


In [ ]:
# Retrieve and save your best automl model.

### YOUR CODE HERE ###
automl_best_run = autml_run.get_best_child()
print(automl_best_run)

StatementMeta(, , , Waiting, , Waiting)